In [0]:

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pyspark.sql.functions as F
from functools import reduce
from operator import or_


# ============================================================
# 2. DEFINE PRIMARY KEY VALIDATION FUNCTION
# ============================================================

def validate_primary_key(dataframe, primary_key, dataset_name):

    # Check whether any candidate key column contains NULL
    null_conditions = [
        F.col(column).isNull()
        for column in primary_key
    ]

    combined_null_condition = reduce(or_, null_conditions)

    null_count = (
        dataframe
        .filter(combined_null_condition)
        .count()
    )

    # Check duplicate groups using all candidate key columns
    duplicate_count = (
        dataframe
        .groupBy(*primary_key)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    # Determine validation status
    if null_count == 0 and duplicate_count == 0:
        status = "PASS"
        print(
            f"PASS: {primary_key} satisfies NULL and uniqueness "
            f"checks in the {dataset_name} dataset."
        )
    else:
        status = "FAIL"
        print(
            f"FAIL: {primary_key} does not satisfy NULL and "
            f"uniqueness checks in the {dataset_name} dataset."
        )

    # Store the result in a dictionary
    result = {
        "dataset_name": dataset_name,
        "primary_key": ", ".join(primary_key),
        "null_count": null_count,
        "duplicate_count": duplicate_count,
        "status": status
    }

    return result


# ============================================================
# 3. DEFINE RAW DATASET PATHS
# ============================================================

common_path = "/Volumes/retailpulse/dev/olist_raw"

datasets = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "product_category_name_translation":
        "product_category_name_translation.csv"
}


# ============================================================
# 4. LOAD ALL DATASETS INTO A DATAFRAME DICTIONARY
# ============================================================

dataframes = {}

for dataset_name, file_name in datasets.items():

    file_path = f"{common_path}/{file_name}"

    dataframes[dataset_name] = (
        spark.read
        .format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(file_path)
    )

print(f"Loaded {len(dataframes)} datasets successfully.")


# ============================================================
# 5. DISPLAY DATASET RECORD COUNTS
# ============================================================

for dataset_name, dataframe in dataframes.items():
    print(f"{dataset_name}: {dataframe.count()} rows")


# ============================================================
# 6. CONFIGURE CANDIDATE PRIMARY KEYS
# ============================================================

primary_key_config = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_reviews": ["review_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "product_category_name_translation": [
        "product_category_name"
    ],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"]
}


# ============================================================
# 7. AUTOMATE PRIMARY KEY VALIDATION
# ============================================================

primary_key_validation_results = []

for dataset_name, primary_key in primary_key_config.items():

    dataframe = dataframes[dataset_name]

    result = validate_primary_key(
        dataframe=dataframe,
        primary_key=primary_key,
        dataset_name=dataset_name
    )

    primary_key_validation_results.append(result)


# ============================================================
# 8. CREATE CONSOLIDATED VALIDATION REPORT
# ============================================================

validation_df = spark.createDataFrame(
    primary_key_validation_results
)

display(validation_df)
